In [38]:
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.mask import mask
import pandas as pd

In [11]:


# 1️⃣ Charger le raster CDL d'Arkansas
cdl_path = "CDL_2021_06.tif"
with rasterio.open(cdl_path) as cdl_src:
    cdl_data = cdl_src.read(1)
    cdl_meta = cdl_src.meta.copy()
    cdl_bounds = cdl_src.bounds
    cdl_crs = cdl_src.crs

# 2️⃣ Charger le raster national Confidence Layer
conf_path = "2021_30m_confidence_layer.tif"
with rasterio.open(conf_path) as conf_src:
    
    # ⚠️ projeter le raster national si nécessaire
    if conf_src.crs != cdl_crs:
        raise ValueError("Le CRS du Confidence Layer doit correspondre au CDL")
    
    # 3️⃣ Découper uniquement la zone Arkansas
    window = rasterio.windows.from_bounds(
        cdl_bounds.left, cdl_bounds.bottom, cdl_bounds.right, cdl_bounds.top,
        transform=conf_src.transform
    )
    conf_data = conf_src.read(1, window=window)

# 4️⃣ Sauvegarder un GeoTIFF avec 2 couches : CDL + Confidence
out_meta = cdl_meta.copy()
out_meta.update({
    "count": 2  # 2 couches
})

with rasterio.open("california_cdl_confidence.tif", "w", **out_meta) as dst:
    dst.write(cdl_data, 1)       # couche 1 = CDL
    dst.write(conf_data, 2)      # couche 2 = Confidence

print("✅ Nouveau fichier multi-couches créé !")

✅ Nouveau fichier multi-couches créé !


In [21]:
with rasterio.open("CDL_2021_05.tif") as src:
    print(f"Nombre de bandes : {src.count}")
    print(f"Description des bandes : {src.descriptions}")
    data_cdl = src.read() 
    


     

    
print(data_cdl.shape)


Nombre de bandes : 1
Description des bandes : ('Layer_1',)
(1, 13440, 14847)


In [ ]:
import ee

# Authentification
ee.Authenticate(auth_mode='notebook')



In [17]:
import ee
ee.Initialize(project='plenary-glass-467416-q1')

# Arkansas zone 1
zone_ark_1 = ee.Geometry.Rectangle([-91.30, 34.90, -91.10, 35.10])
# Arkansas zone 2
zone_ark_2 = ee.Geometry.Rectangle([-91.60, 34.20, -91.40, 34.40])
# Californie zone 1
zone_cal_1 = ee.Geometry.Rectangle([-121.90, 39.60, -121.70, 39.80])
# Californie zone 2
zone_cal_2 = ee.Geometry.Rectangle([-121.70, 38.40, -121.50, 38.60])

print("Zones définies ✓")

Zones définies ✓


In [18]:
BANDES = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

def masquer_nuages(image):
    scl    = image.select('SCL')
    masque = (scl.eq(4).Or(scl.eq(5))
                        .Or(scl.eq(6))
                        .Or(scl.eq(11)))
    return image.updateMask(masque).select(BANDES)

def mediane_10jours(i, zone):
    debut = ee.Date('2021-01-01').advance(i * 10, 'day')
    fin   = ee.Date('2021-01-01').advance((i + 1) * 10, 'day')
    col   = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(zone)
             .filterDate(debut, fin)
             .map(masquer_nuages))
    mediane      = col.median().unmask(0)
    est_manquant = (ee.Image(col.size().eq(0))
                    .rename(f'msk_t{i:02d}')
                    .toByte())
    noms = [f'{b}_t{i:02d}' for b in BANDES]
    return mediane.rename(noms).addBands(est_manquant)

def construire_stack(zone):
    images = [mediane_10jours(i, zone) for i in range(36)]
    stack  = images[0]
    for img in images[1:]:
        stack = stack.addBands(img)
    return stack

def exporter_zone(zone, nom):
    stack = construire_stack(zone)
    tache = ee.batch.Export.image.toDrive(
        image          = stack.clip(zone).toFloat(),
        description    = f'S2_{nom}',
        folder         = 'MCTNet_S2',
        fileNamePrefix = f'S2_{nom}',
        region         = zone,
        scale          = 30,          # ← 30m comme ton CDL
        crs            = 'EPSG:4326',
        maxPixels      = 1e13,
        fileFormat     = 'GeoTIFF'
    )
    tache.start()
    print(f"Export {nom} lancé — état : {tache.status()['state']}")
    return tache

# Lancer les 4 exports
tache_ark1 = exporter_zone(zone_ark_1, 'Arkansas_zone1')
tache_ark2 = exporter_zone(zone_ark_2, 'Arkansas_zone2')
tache_cal1 = exporter_zone(zone_cal_1, 'California_zone1')
tache_cal2 = exporter_zone(zone_cal_2, 'California_zone2')

Export Arkansas_zone1 lancé — état : READY
Export Arkansas_zone2 lancé — état : READY
Export California_zone1 lancé — état : READY
Export California_zone2 lancé — état : READY


In [ ]:
print("Arkansas   :", tache_ark1.status()['state'])
print("Californie :", tache_cal1.status()['state'])
print("Arkansas   :", tache_ark2.status()['state'])
print("Californie :", tache_cal2.status()['state'])

Arkansas   : COMPLETED
Californie : COMPLETED
Arkansas   : RUNNING
Californie : COMPLETED


In [2]:
with rasterio.open("S2_Arkansas_zone1.tif") as src:
    print(f"Nombre de bandes : {src.count}")
    print(f"Description des bandes : {src.descriptions}")
    data_a = src.read(11)  

print(data_a.shape) 

Nombre de bandes : 396
Description des bandes : ('B2_t00', 'B3_t00', 'B4_t00', 'B5_t00', 'B6_t00', 'B7_t00', 'B8_t00', 'B8A_t00', 'B11_t00', 'B12_t00', 'msk_t00', 'B2_t01', 'B3_t01', 'B4_t01', 'B5_t01', 'B6_t01', 'B7_t01', 'B8_t01', 'B8A_t01', 'B11_t01', 'B12_t01', 'msk_t01', 'B2_t02', 'B3_t02', 'B4_t02', 'B5_t02', 'B6_t02', 'B7_t02', 'B8_t02', 'B8A_t02', 'B11_t02', 'B12_t02', 'msk_t02', 'B2_t03', 'B3_t03', 'B4_t03', 'B5_t03', 'B6_t03', 'B7_t03', 'B8_t03', 'B8A_t03', 'B11_t03', 'B12_t03', 'msk_t03', 'B2_t04', 'B3_t04', 'B4_t04', 'B5_t04', 'B6_t04', 'B7_t04', 'B8_t04', 'B8A_t04', 'B11_t04', 'B12_t04', 'msk_t04', 'B2_t05', 'B3_t05', 'B4_t05', 'B5_t05', 'B6_t05', 'B7_t05', 'B8_t05', 'B8A_t05', 'B11_t05', 'B12_t05', 'msk_t05', 'B2_t06', 'B3_t06', 'B4_t06', 'B5_t06', 'B6_t06', 'B7_t06', 'B8_t06', 'B8A_t06', 'B11_t06', 'B12_t06', 'msk_t06', 'B2_t07', 'B3_t07', 'B4_t07', 'B5_t07', 'B6_t07', 'B7_t07', 'B8_t07', 'B8A_t07', 'B11_t07', 'B12_t07', 'msk_t07', 'B2_t08', 'B3_t08', 'B4_t08', 'B5_t08',

In [ ]:


with rasterio.open("S2_Arkansas_zone1.tif") as src:
    print(f"Nombre de bandes : {src.count}")
    
    data = src.read()  # lire toutes les couches
    profile = src.profile

new_layers = []

for i in range(0, data.shape[0], 11):
    data_block = data[i:i+10]      # 10 couches de données
    mask = data[i+10]              # 11e couche (masque)
    
    # appliquer le masque
    masked_block = np.where(mask == 0, data_block, 0)
    
    new_layers.append(masked_block)

# concaténer
result = np.concatenate(new_layers, axis=0)

# sauvegarde
profile.update(count=result.shape[0])

with rasterio.open("S2_Arkansas_zone1_updated.tif", "w", **profile) as dst:
    dst.write(result)

Nombre de bandes : 396


In [14]:


with rasterio.open("S2_Arkansas_zone2.tif") as src:
    print(f"Nombre de bandes : {src.count}")
    
    data = src.read()  # lire toutes les couches
    profile = src.profile

new_layers = []

for i in range(0, data.shape[0], 11):
    data_block = data[i:i+10]      # 10 couches de données
    mask = data[i+10]              # 11e couche (masque)
    
    # appliquer le masque
    masked_block = np.where(mask == 0, data_block, 0)
    
    new_layers.append(masked_block)

# concaténer
result = np.concatenate(new_layers, axis=0)

# sauvegarde
profile.update(count=result.shape[0])

with rasterio.open("S2_Arkansas_zone2_updated.tif", "w", **profile) as dst:
    dst.write(result)

Nombre de bandes : 396


In [13]:
with rasterio.open("S2_Arkansas_zone1_updated.tif") as src:
    print(f"Nombre de bandes : {src.count}")
    
    data = src.read()  # lire toutes les couches

print(data.shape)
print(np.unique(data[0]))    

Nombre de bandes : 360
(360, 743, 743)
[   0.    45.5   47.5 ... 5827.  5906.  6309. ]


In [15]:
with rasterio.open("S2_Arkansas_zone2_updated.tif") as src:
    print(f"Nombre de bandes : {src.count}")
    
    data = src.read()  # lire toutes les couches

print(data.shape)
print(np.unique(data[0])) 

Nombre de bandes : 360
(360, 743, 743)
[   0.   33.   34. ... 3910. 4292. 4596.]


In [19]:
print("Sentinel shape :", data.shape)
print("CDL shape :", data_cdl.shape)

Sentinel shape : (360, 743, 743)
CDL shape : (2, 13440, 14847)


## 1.appliquer les cdl sur les deux zones de l'arkansas


In [ ]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import reproject, Resampling
import numpy as np

# 1️⃣ Coordonnées exactes de la zone
xmin, ymin, xmax, ymax = -91.30, 34.90, -91.10, 35.10

# 2️⃣ Charger le raster Sentinel-2
sentinel_path = "S2_Arkansas_zone1_updated.tif"
with rasterio.open(sentinel_path) as src:
    window = from_bounds(xmin, ymin, xmax, ymax, transform=src.transform)
    sentinel_data = src.read(window=window)  # (360, H, W)
    meta = src.meta.copy()
    meta.update({
        "height": sentinel_data.shape[1],
        "width": sentinel_data.shape[2],
        "transform": src.window_transform(window)
    })
    sentinel_crs = src.crs

# 3️⃣ Charger le raster CDL national
cdl_path = "arkansas_cdl_confidence.tif"
with rasterio.open(cdl_path) as cdl_src:
    # Créer un tableau vide pour les 2 bandes CDL alignées avec le Sentinel
    aligned_cdl = np.zeros((2, sentinel_data.shape[1], sentinel_data.shape[2]), dtype=cdl_src.dtypes[0])
    
    for i in range(2):
        reproject(
            source=rasterio.band(cdl_src, i+1),
            destination=aligned_cdl[i],
            src_transform=cdl_src.transform,
            src_crs=cdl_src.crs,
            dst_transform=meta["transform"],
            dst_crs=sentinel_crs,
            resampling=Resampling.nearest
        )

# 4️⃣ Fusionner Sentinel + CDL
result = np.concatenate([sentinel_data, aligned_cdl], axis=0)  # (362, H, W)

# 5️⃣ Mettre à jour metadata et sauvegarder
meta.update({"count": result.shape[0]})
output_path = "arkansas_zone1_with_cdl.tif"
with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(result)

print("✅ Fichier multi-couches créé avec CDL ")

✅ Fichier multi-couches créé avec CDL projeté sur EPSG:4326 !


In [ ]:
with rasterio.open("arkansas_zone1_with_cdl.tif") as src:
       data_a_zone1=src.read()

print(data.shape)       

(362, 742, 742)


In [27]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import reproject, Resampling
import numpy as np

# 1️⃣ Coordonnées exactes de la zone
xmin, ymin, xmax, ymax = -91.60, 34.20, -91.40, 34.40

# 2️⃣ Charger le raster Sentinel-2
sentinel_path = "S2_Arkansas_zone2_updated.tif"
with rasterio.open(sentinel_path) as src:
    window = from_bounds(xmin, ymin, xmax, ymax, transform=src.transform)
    sentinel_data = src.read(window=window)  # (360, H, W)
    meta = src.meta.copy()
    meta.update({
        "height": sentinel_data.shape[1],
        "width": sentinel_data.shape[2],
        "transform": src.window_transform(window)
    })
    sentinel_crs = src.crs

# 3️⃣ Charger le raster CDL national
cdl_path = "arkansas_cdl_confidence.tif"
with rasterio.open(cdl_path) as cdl_src:
    # Créer un tableau vide pour les 2 bandes CDL alignées avec le Sentinel
    aligned_cdl = np.zeros((2, sentinel_data.shape[1], sentinel_data.shape[2]), dtype=cdl_src.dtypes[0])
    
    for i in range(2):
        reproject(
            source=rasterio.band(cdl_src, i+1),
            destination=aligned_cdl[i],
            src_transform=cdl_src.transform,
            src_crs=cdl_src.crs,
            dst_transform=meta["transform"],
            dst_crs=sentinel_crs,
            resampling=Resampling.nearest
        )

# 4️⃣ Fusionner Sentinel + CDL
result = np.concatenate([sentinel_data, aligned_cdl], axis=0)  # (362, H, W)

# 5️⃣ Mettre à jour metadata et sauvegarder
meta.update({"count": result.shape[0]})
output_path = "arkansas_zone2_with_cdl.tif"
with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(result)

print("✅ Fichier multi-couches créé avec CDL ")

✅ Fichier multi-couches créé avec CDL 


In [ ]:
with rasterio.open("arkansas_zone2_with_cdl.tif") as src:
       data_a_zone2=src.read()

print(data_a_zone2.shape)       


(362, 742, 742)


## charger et appliquer esa worldcover 2021

In [30]:

import ee
ee.Initialize(project='plenary-glass-467416-q1')
# Définir les 4 zones
zone_ark_1 = ee.Geometry.Rectangle([-91.30, 34.90, -91.10, 35.10])
zone_ark_2 = ee.Geometry.Rectangle([-91.60, 34.20, -91.40, 34.40])
zone_cal_1 = ee.Geometry.Rectangle([-121.90, 39.60, -121.70, 39.80])
zone_cal_2 = ee.Geometry.Rectangle([-121.70, 38.40, -121.50, 38.60])

# Charger ESA WorldCover 2021
worldcover = (ee.ImageCollection('ESA/WorldCover/v200')
              .first()
              .select('Map'))

def exporter_worldcover(zone, nom):
    tache = ee.batch.Export.image.toDrive(
        image          = worldcover.clip(zone).toByte(),
        description    = f'WC_{nom}',
        folder         = 'MCTNet_S2',
        fileNamePrefix = f'WorldCover_{nom}',
        region         = zone,
        scale          = 30,        # même résolution que CDL
        crs            = 'EPSG:4326',
        maxPixels      = 1e13,
        fileFormat     = 'GeoTIFF'
    )
    tache.start()
    print(f"Export WorldCover {nom} lancé — état : {tache.status()['state']}")
    return tache

# Lancer les 4 exports
tache_ark1 = exporter_worldcover(zone_ark_1, 'Arkansas_zone1_esa')
tache_ark2 = exporter_worldcover(zone_ark_2, 'Arkansas_zone2_esa')
tache_cal1 = exporter_worldcover(zone_cal_1, 'California_zone1_esa')
tache_cal2 = exporter_worldcover(zone_cal_2, 'California_zone2_esa')

Export WorldCover Arkansas_zone1_esa lancé — état : READY
Export WorldCover Arkansas_zone2_esa lancé — état : READY
Export WorldCover California_zone1_esa lancé — état : READY
Export WorldCover California_zone2_esa lancé — état : READY


In [34]:
import os
from rasterio.warp import reproject, Resampling


def appliquer_masque_agricole(path_362, path_wc, path_output, nom_zone):
    """
    - Garder uniquement les pixels agricoles (WorldCover classe 40)
    - Marquer les pixels non agricoles par -1
    - Enlever la couche WorldCover
    Résultat : 362 couches avec pixels non agricoles = -1
    """
    print(f"\n=== {nom_zone} ===")
    
    # ── 1. Charger le fichier 362 couches ───────────────────
    with rasterio.open(path_362) as src:
        data_362  = src.read().astype(np.float32)  # (362, H, W)
        transform = src.transform
        crs       = src.crs
        H, W      = src.height, src.width
        meta      = src.meta.copy()
    
    print(f"Fichier chargé : {data_362.shape}")
    
    # ── 2. Charger et reprojeter WorldCover ─────────────────
    with rasterio.open(path_wc) as src_wc:
        wc_reproj = np.zeros((H, W), dtype=np.uint8)
        reproject(
            source        = rasterio.band(src_wc, 1),
            destination   = wc_reproj,
            src_transform = src_wc.transform,
            src_crs       = src_wc.crs,
            dst_transform = transform,
            dst_crs       = crs,
            dst_shape     = (H, W),
            resampling    = Resampling.nearest
        )
    
    # ── 3. Créer le masque agricole ──────────────────────────
    masque_agricole     = wc_reproj == 40   # True = agricole
    masque_non_agricole = ~masque_agricole  # True = non agricole
    
    print(f"Pixels agricoles     : {masque_agricole.sum():,}")
    print(f"Pixels non agricoles : {masque_non_agricole.sum():,}")
    
    # ── 4. Appliquer -1 sur les zones non agricoles ──────────
    # Pour chaque bande, mettre -1 où ce n'est pas agricole
    data_masque = data_362.copy()
    data_masque[:, masque_non_agricole] = -1
    
    print(f"Pixels non agricoles marqués -1 ✓")
    
    # ── 5. Sauvegarder (362 couches sans WorldCover) ─────────
    meta.update(count=362, dtype='float32')
    
    os.makedirs(os.path.dirname(path_output) or '.', exist_ok=True)
    
    with rasterio.open(path_output, 'w', **meta) as dst:
        dst.write(data_masque)
    
    print(f"Sauvegardé : {path_output} ✓")
    print(f"Structure finale :")
    print(f"  Bandes 1-360 : Sentinel-2 (-1 si non agricole)")
    print(f"  Bande  361   : CDL classes (-1 si non agricole)")
    print(f"  Bande  362   : CDL confidence (-1 si non agricole)")


appliquer_masque_agricole(
    path_362   = 'Arkansas_zone1_with_cdl.tif',
    path_wc    = 'WorldCover_Arkansas_zone1_esa.tif',
    path_output= 'MCTNet_Final/Arkansas_zone1_masked.tif',
    nom_zone   = 'Arkansas zone 1'
)

appliquer_masque_agricole(
    path_362   = 'Arkansas_zone2_with_cdl.tif',
    path_wc    = 'WorldCover_Arkansas_zone2_esa.tif',
    path_output= 'MCTNet_Final/Arkansas_zone2_masked.tif',
    nom_zone   = 'Arkansas zone 2'
)




=== Arkansas zone 1 ===
Fichier chargé : (362, 742, 742)
Pixels agricoles     : 314,800
Pixels non agricoles : 235,764
Pixels non agricoles marqués -1 ✓
Sauvegardé : MCTNet_Final/Arkansas_zone1_masked.tif ✓
Structure finale :
  Bandes 1-360 : Sentinel-2 (-1 si non agricole)
  Bande  361   : CDL classes (-1 si non agricole)
  Bande  362   : CDL confidence (-1 si non agricole)

=== Arkansas zone 2 ===
Fichier chargé : (362, 742, 742)
Pixels agricoles     : 351,437
Pixels non agricoles : 199,127
Pixels non agricoles marqués -1 ✓
Sauvegardé : MCTNet_Final/Arkansas_zone2_masked.tif ✓
Structure finale :
  Bandes 1-360 : Sentinel-2 (-1 si non agricole)
  Bande  361   : CDL classes (-1 si non agricole)
  Bande  362   : CDL confidence (-1 si non agricole)


In [35]:
with rasterio.open("MCTNet_Final/Arkansas_zone1_masked.tif") as src:
    data=src.read(1)

print(np.unique(data))    

[-1.0000e+00  0.0000e+00  5.6500e+01 ...  2.3360e+03  2.3660e+03
  2.9265e+03]


## appliquer la couche de confiance >=95%


In [36]:

import os

def filtrer_confidence(path_masked, path_output, nom_zone):
    """
    - Garder uniquement les pixels avec CDL confidence >= 95%
    - Marquer les pixels confidence < 95% par -1
    - Enlever la couche confidence (bande 362)
    Résultat : 361 couches
      - bandes 1-360 : Sentinel-2
      - bande  361   : CDL classes
    """
    print(f"\n=== {nom_zone} ===")
    
    # ── 1. Charger le fichier 362 couches ───────────────────
    with rasterio.open(path_masked) as src:
        data     = src.read().astype(np.float32)  # (362, H, W)
        meta     = src.meta.copy()
        H, W     = src.height, src.width
    
    print(f"Fichier chargé : {data.shape}")
    
    # ── 2. Extraire la couche confidence (bande 362) ─────────
    confidence = data[361]   # index 361 = bande 362
    cdl        = data[360]   # index 360 = bande 361 (CDL classes)
    sentinel   = data[:360]  # bandes 1-360 = Sentinel-2
    
    print(f"Confidence min/max : {confidence.min():.1f} / "
          f"{confidence.max():.1f}")
    
    # ── 3. Créer le masque confidence ────────────────────────
    # Ignorer les pixels déjà à -1 (non agricoles)
    masque_deja_invalide = data[0] == -1        # pixels non agricoles
    masque_conf_ok       = confidence >= 95     # confidence suffisante
    masque_conf_faible   = (~masque_conf_ok) & (~masque_deja_invalide)
    
    print(f"Pixels déjà invalides (non agricoles) : "
          f"{masque_deja_invalide.sum():,}")
    print(f"Pixels confidence < 95%               : "
          f"{masque_conf_faible.sum():,}")
    print(f"Pixels valides (agricoles + conf>=95) : "
          f"{(masque_conf_ok & ~masque_deja_invalide).sum():,}")
    
    # ── 4. Appliquer -1 sur pixels confidence < 95% ──────────
    sentinel_filtre = sentinel.copy()
    cdl_filtre      = cdl.copy()
    
    sentinel_filtre[:, masque_conf_faible] = -1
    cdl_filtre[masque_conf_faible]         = -1
    
    # ── 5. Empiler sans la couche confidence ─────────────────
    # 360 bandes S2 + 1 bande CDL = 361 bandes
    data_361 = np.concatenate([
        sentinel_filtre,              # (360, H, W)
        cdl_filtre[np.newaxis, :, :]  # (1,   H, W)
    ], axis=0)                        # (361, H, W)
    
    print(f"Stack final : {data_361.shape}")
    
    # ── 6. Sauvegarder ──────────────────────────────────────
    meta.update(count=361, dtype='float32')
    
    os.makedirs(os.path.dirname(path_output) or '.', exist_ok=True)
    
    with rasterio.open(path_output, 'w', **meta) as dst:
        dst.write(data_361)
    
    print(f"Sauvegardé : {path_output} ✓")
    print(f"Structure finale :")
    print(f"  Bandes 1-360 : Sentinel-2")
    print(f"  Bande  361   : CDL classes")
    print(f"  pixels invalides (non agri ou conf<95%) → -1")

# ── Appliquer sur les 4 zones ────────────────────────────────
filtrer_confidence(
    path_masked = 'MCTNet_Final/Arkansas_zone1_masked.tif',
    path_output = 'MCTNet_Final/Arkansas_zone1_final.tif',
    nom_zone    = 'Arkansas zone 1'
)

filtrer_confidence(
    path_masked = 'MCTNet_Final/Arkansas_zone2_masked.tif',
    path_output = 'MCTNet_Final/Arkansas_zone2_final.tif',
    nom_zone    = 'Arkansas zone 2'
)




=== Arkansas zone 1 ===
Fichier chargé : (362, 742, 742)
Confidence min/max : -1.0 / 100.0
Pixels déjà invalides (non agricoles) : 235,764
Pixels confidence < 95%               : 166,744
Pixels valides (agricoles + conf>=95) : 148,056
Stack final : (361, 742, 742)
Sauvegardé : MCTNet_Final/Arkansas_zone1_final.tif ✓
Structure finale :
  Bandes 1-360 : Sentinel-2
  Bande  361   : CDL classes
  pixels invalides (non agri ou conf<95%) → -1

=== Arkansas zone 2 ===
Fichier chargé : (362, 742, 742)
Confidence min/max : -1.0 / 100.0
Pixels déjà invalides (non agricoles) : 199,127
Pixels confidence < 95%               : 151,961
Pixels valides (agricoles + conf>=95) : 199,476
Stack final : (361, 742, 742)
Sauvegardé : MCTNet_Final/Arkansas_zone2_final.tif ✓
Structure finale :
  Bandes 1-360 : Sentinel-2
  Bande  361   : CDL classes
  pixels invalides (non agri ou conf<95%) → -1


## convertir en fichier csv


In [39]:


BANDES = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

def tif_vers_csv(path_final, nom_zone, output_dir='MCTNet_CSV'):
    """
    Convertir le fichier final 361 bandes en CSV.
    - Ignorer les pixels à -1 (non agricoles ou conf<95%)
    - Chaque ligne = 1 pixel valide
    - 360 colonnes spectrales + 1 colonne label
    """
    print(f"\n=== {nom_zone} ===")
    
    # ── 1. Charger le fichier ────────────────────────────────
    with rasterio.open(path_final) as src:
        data = src.read().astype(np.float32)  # (361, H, W)
        H, W = src.height, src.width
    
    print(f"Fichier chargé : {data.shape}")
    
    # ── 2. Séparer Sentinel-2 et CDL ─────────────────────────
    sentinel = data[:360]   # (360, H, W)
    cdl      = data[360]    # (H, W)
    
    # ── 3. Garder uniquement les pixels valides (≠ -1) ───────
    masque_valide = cdl != -1
    print(f"Pixels valides : {masque_valide.sum():,}")
    print(f"Pixels invalides (-1) : {(~masque_valide).sum():,}")
    
    if masque_valide.sum() == 0:
        print(f"⚠️ Aucun pixel valide dans {nom_zone} !")
        return None
    
    rows, cols = np.where(masque_valide)
    n_pixels   = len(rows)
    
    # ── 4. Extraire les features ─────────────────────────────
    # sentinel shape : (360, H, W)
    # on veut       : (n_pixels, 360)
    features = sentinel[:, rows, cols]   # (360, n_pixels)
    features = features.T                # (n_pixels, 360)
    
    # ── 5. Noms des colonnes ─────────────────────────────────
    cols_names = [f'{b}_t{t:02d}' 
                  for t in range(36) 
                  for b in BANDES]
    
    # ── 6. Construire le DataFrame ───────────────────────────
    df          = pd.DataFrame(features, columns=cols_names)
    df['label'] = cdl[rows, cols].astype(int)
    
    print(f"Distribution des cultures :")
    print(df['label'].value_counts().to_string())
    
    # ── 7. Sauvegarder ──────────────────────────────────────
    os.makedirs(output_dir, exist_ok=True)
    path_out = f'{output_dir}/{nom_zone}.csv'
    df.to_csv(path_out, index=False)
    
    print(f"\nCSV sauvegardé : {path_out}")
    print(f"Shape : {df.shape}")  # (n_pixels, 361)
    
    return df

# ── Appliquer sur les 4 zones ────────────────────────────────
df_ark1 = tif_vers_csv('MCTNet_Final/Arkansas_zone1_final.tif',
                        'Arkansas_zone1')

df_ark2 = tif_vers_csv('MCTNet_Final/Arkansas_zone2_final.tif',
                        'Arkansas_zone2')




=== Arkansas_zone1 ===
Fichier chargé : (361, 742, 742)
Pixels valides : 148,056
Pixels invalides (-1) : 402,508
Distribution des cultures :
label
5      70490
3      56494
1      19150
121      465
26       414
122      263
61       195
4        171
240      120
2        111
123      109
124       39
190       33
37         2

CSV sauvegardé : MCTNet_CSV/Arkansas_zone1.csv
Shape : (148056, 361)

=== Arkansas_zone2 ===
Fichier chargé : (361, 742, 742)
Pixels valides : 199,476
Pixels invalides (-1) : 351,088
Distribution des cultures :
label
5      106011
3       60792
1       29310
24        923
122       837
121       522
61        480
123       388
190        86
124        63
240        56
111         6
2           2

CSV sauvegardé : MCTNet_CSV/Arkansas_zone2.csv
Shape : (199476, 361)


## Echantillonnage


In [44]:
def echantillonner_final(df_zone1, df_zone2, nom_etat, 
                          output_dir='MCTNet_CSV', seed=42):
    """
    Fusionner les 2 zones 
    
    """
    rng = np.random.default_rng(seed)
    
    # Fusionner
    df = pd.concat([df_zone1, df_zone2], ignore_index=True)
    print(f"\n=== {nom_etat} ===")
    print(f"Total pixels disponibles : {len(df):,}")
    
    
    
    # Sauvegarder
    path_out = f'{output_dir}/{nom_etat}_final.csv'
    df.to_csv(path_out, index=False)
    print(f"\nCSV final sauvegardé : {path_out} ✓")
    
    return df

# Arkansas
df_arkansas = echantillonner_final(df_ark1, df_ark2, 'Arkansas')




=== Arkansas ===
Total pixels disponibles : 347,532

CSV final sauvegardé : MCTNet_CSV/Arkansas_final.csv ✓


In [47]:


# Lire le fichier CSV
df = pd.read_csv("MCTNet_CSV/Arkansas_final.csv")

# Prendre un échantillon aléatoire de 10000 lignes
sample_df = df.sample(n=10000, random_state=42)

# Sauvegarder le résultat
sample_df.to_csv("sample_10000.csv", index=False)

In [49]:
df = pd.read_csv("MCTNet_CSV/Arkansas_final.csv")

print(len(df.columns))

361
